In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


feature details:
|Feature|Mark|
|-------|----|
|AT|temprature|
|V|vacum|
|AP|pressure|
|RH|humidity|
|PE|produced energy (target)|

In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

* no null values found

#### Data and Split

In [5]:
X = df.drop("PE", axis=1)
y = df["PE"]

y.head()

0    480.48
1    445.75
2    438.76
3    453.09
4    464.43
Name: PE, dtype: float64

In [6]:
#Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

df.shape

(9568, 5)

#### Scaling

In [7]:
#Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

X_test_scaled

array([[ 1.36255441,  0.24305848, -1.27798491, -1.04586379],
       [ 0.82147733,  1.36986442, -0.74695303,  0.31071896],
       [-0.247116  , -0.73708132,  1.92304887, -0.14668594],
       ...,
       [-0.67970645, -1.15815506, -0.31652036, -0.05697206],
       [ 1.33136451,  1.34463153, -0.87558808, -0.39000089],
       [-0.26474508, -0.26711887,  0.33984824,  1.14397069]],
      shape=(1914, 4))

#### Tensors

In [8]:
#Tensors
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)


#### Batchs

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

#batch_size defines no of samples for each bathc of data
#shuffle, shuffles dataset before creating batches (to get randomn samples in a batch instead of sequential samples)


### ANN

In [12]:
#Define our ANN Model
class ANN_model(nn.Module):
    def __init__(self):
        super(ANN_model, self).__init__()
        #confusion: why we call super

        #neural network arch
        self.model = nn.Sequential(
            #hidden layer-I
            nn.Linear(X_train.shape[1], 6), #rows, coumns
            #linear: x1w1 + b
            nn.ReLU(),

            #hidden layer-II
            nn.Linear(6, 6),
            nn.ReLU(),
                
            # output layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

import torch.optim as optim

model = ANN_model()
#loss, optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

#### Train model

NameError: name 'train_loader' is not defined